# ShopEase — Gradio demo for Google Colab
Runs the ShopEase catalogue (184 products), cart, checkout, orders, support policies and a chatbot placeholder in a Gradio UI.
**Steps:** run the cells in order. When asked, upload `shopease.zip` (the project you downloaded).
Demo logins: `user@shopease.com` / `user1234` (has sample orders), `admin@shopease.com` / `admin1234`.

In [ ]:
!pip install -q gradio pandas requests mongomock-motor motor bcrypt pyjwt "pydantic[email]"

In [ ]:
# Upload shopease.zip (skipped if the folder already exists)
import os, zipfile
if not os.path.exists("shopease/backend"):
    from google.colab import files
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall("shopease")
print(os.listdir("shopease"))

In [ ]:
# Load project data using the project's own seed generators (no MongoDB needed)
import sys, os, json, glob, random, uuid, datetime as dt
os.environ["MONGO_URL"] = "mock"
sys.path.insert(0, os.path.abspath("shopease/backend"))
import seed
from app.security import hash_pw, check_pw

PRODUCTS = seed.products()
PRODUCTS += seed.curated_products(len(PRODUCTS) + 1)
seed.policies()                                   # (re)writes the policy JSON files
BY_ID = {p["product_id"]: p for p in PRODUCTS}
CATEGORIES = sorted({p["category"] for p in PRODUCTS})
POLICIES = {json.load(open(f))["title"]: json.load(open(f)) for f in sorted(glob.glob("shopease/backend/data/policies/*.json"))}

USERS = {
    "user@shopease.com": {"user_id": "U0001", "name": "Demo User", "pw": hash_pw("user1234"), "role": "customer"},
    "admin@shopease.com": {"user_id": "U0002", "name": "Admin", "pw": hash_pw("admin1234"), "role": "admin"},
}
COURIERS = ["Delhivery", "BlueDart", "Ecom Express"]

def make_order(uid, items, ship, pay, days_ago, method="UPI", address=None):
    return {"order_id": "ORD" + uuid.uuid4().hex[:8].upper(), "tracking_id": "TRK" + str(random.randint(10**9, 10**10 - 1)), "user_id": uid,
            "products": [{"title": p["title"], "price": p["price"], "quantity": q} for p, q in items],
            "total": round(sum(p["price"] * q for p, q in items), 2), "payment_method": method, "payment_status": pay, "shipping_status": ship,
            "courier_partner": random.choice(COURIERS), "delivery_date": (dt.date.today() + dt.timedelta(days=5 - days_ago)).isoformat(), "address": address or {}}

ORDERS = [make_order("U0001", [(p, 1) for p in random.sample(PRODUCTS, 2)], s, "paid" if s != "processing" else "pending", d)
          for s, d in [("delivered", 30), ("delivered", 20), ("shipped", 3), ("processing", 0)]]
print(len(PRODUCTS), "products |", len(POLICIES), "policies |", len(ORDERS), "sample orders")

In [ ]:
import pandas as pd, requests, gradio as gr

COLS = ["ID", "Title", "Brand", "Category", "Price (Rs)", "Off %", "Rating", "Stock"]
def table(items):
    return pd.DataFrame([[p["product_id"], p["title"], p["brand"], p["category"], p["price"], p["discount"], p["rating"], p["stock"]] for p in items], columns=COLS)

def search(q, cat, min_rating, max_price, sort):
    q = (q or "").lower()
    r = [p for p in PRODUCTS if (not q or q in f"{p['title']} {p['brand']} {p['category']} {p['description']}".lower())
         and (cat == "All" or p["category"] == cat) and p["rating"] >= min_rating and p["price"] <= max_price]
    key = {"Price: low to high": (lambda p: p["price"], False), "Price: high to low": (lambda p: p["price"], True), "Top rated": (lambda p: p["rating"], True)}.get(sort)
    if key: r.sort(key=key[0], reverse=key[1])
    return table(r[:100]), f"{len(r)} products found"

def suggest(q):
    return [p["title"] for p in PRODUCTS if q and q.lower() in p["title"].lower()][:6]

def pick(df, evt: gr.SelectData):
    return df.iloc[evt.index[0]]["ID"]

def detail(pid):
    p = BY_ID.get((pid or "").strip().upper())
    if not p: return "Enter a valid product ID (e.g. P0001).", []
    specs = "\n".join(f"- **{k}:** {v}" for k, v in p["specifications"].items())
    revs = "\n".join(f"- **{r['user']}** ({r['rating']}★): {r['comment']}" for r in p["reviews"])
    sim = ", ".join(f"{s['title']} ({s['product_id']})" for s in [x for x in PRODUCTS if x["category"] == p["category"] and x is not p][:4])
    md = (f"## {p['title']}\n{p['brand']} · Sold by {p['seller']}\n\n**Rs {p['price']:,.0f}**  ~~Rs {p['mrp']:,}~~  ({p['discount']}% off) · {p['rating']}★\n\n{p['description']}\n\n"
          f"Delivery in {p['shipping_time']} · Warranty: {p['warranty']} · {'In stock: ' + str(p['stock']) if p['stock'] else 'Out of stock'}\n\n"
          f"### Specifications\n{specs}\n\n### Reviews\n{revs}\n\n### Similar products\n{sim}")
    return md, p["images"]

# ---- cart ----
def add_to_cart(pid, qty, cart):
    p = BY_ID.get((pid or "").strip().upper())
    if not p: return cart, "Enter a valid product ID."
    cart = dict(cart); n = cart.get(p["product_id"], 0) + int(qty)
    if n > p["stock"]: return cart, "Not enough stock."
    cart[p["product_id"]] = n
    return cart, f"Added {int(qty)} × {p['title']}"

def remove_from_cart(pid, cart):
    cart = dict(cart); cart.pop((pid or "").strip().upper(), None); return cart

def cart_view(cart):
    rows = [[pid, BY_ID[pid]["title"], BY_ID[pid]["price"], q, round(BY_ID[pid]["price"] * q, 2)] for pid, q in cart.items()]
    total = sum(r[4] for r in rows); fee = 0 if total >= 499 or not rows else 40
    return pd.DataFrame(rows, columns=["ID", "Title", "Price", "Qty", "Subtotal"]), f"**Items: Rs {total:,.0f} · Delivery: {'Free' if not fee else 'Rs 40'} · Total: Rs {total + fee:,.0f}**"

def checkout(cart, uid, name, line1, city, pin, pay):
    if not uid: return cart, "Please log in on the Account tab first."
    if not cart: return cart, "Your cart is empty."
    if not all([name, line1, city, pin]): return cart, "Please fill in the full address."
    items = [(BY_ID[pid], q) for pid, q in cart.items()]
    for p, q in items: p["stock"] -= q
    o = make_order(uid, items, "processing", "pending" if pay == "COD" else "paid", 0, pay, {"name": name, "line1": line1, "city": city, "pincode": pin})
    ORDERS.append(o)
    return {}, f"Order **{o['order_id']}** placed. Tracking ID: {o['tracking_id']} · Courier: {o['courier_partner']} · Delivery by {o['delivery_date']}"

# ---- account / orders ----
def login(email, password):
    u = USERS.get((email or "").strip().lower())
    if not u or not check_pw(password or "", u["pw"]): return None, "Invalid email or password."
    return u["user_id"], f"Logged in as {u['name']} ({u['role']})"

def logout(): return None, "Logged out."

STEPS = ["processing", "shipped", "out_for_delivery", "delivered"]
def orders_view(uid):
    if not uid: return "Please log in first."
    mine = [o for o in ORDERS if o["user_id"] == uid]
    def fmt(o):
        bar = " → ".join(f"**{s}**" if i == STEPS.index(o["shipping_status"]) else s for i, s in enumerate(STEPS))
        items = "; ".join(f"{p['title']} × {p['quantity']}" for p in o["products"])
        return f"**{o['order_id']}** · Tracking {o['tracking_id']} ({o['courier_partner']})  \nPayment: {o['payment_status']} · Delivery: {o['delivery_date']} · Total Rs {o['total']:,.0f}  \n{items}  \n{bar}\n"
    cur = [fmt(o) for o in mine if o["shipping_status"] != "delivered"]; prev = [fmt(o) for o in mine if o["shipping_status"] == "delivered"]
    return "### Current orders\n" + ("\n".join(cur) or "None") + "\n\n### Previous orders\n" + ("\n".join(prev) or "None")

# ---- support + chat ----
def policy_view(title):
    d = POLICIES[title]; return f"## {d['title']}\n" + "\n".join(f"**{s['heading']}**  \n{s['content']}\n" for s in d["sections"])

CHAT_URL = ""   # e.g. "http://localhost:8000/chat" once your FastAPI POST /chat is running
def chat_fn(message, history):
    if CHAT_URL:
        try: return requests.post(CHAT_URL, json={"message": message}, timeout=30).json()["reply"]
        except Exception as e: return f"Chat backend unreachable: {e}"
    return "Our AI assistant is coming soon. Meanwhile, see the Support tab for policies."   # TODO: plug in your RAG chatbot

In [ ]:
import gradio as gr

with gr.Blocks(title="ShopEase", theme=gr.themes.Soft(primary_hue="teal")) as demo:
    gr.Markdown("# 🛍️ ShopEase")
    cart, uid = gr.State({}), gr.State(None)

    with gr.Tab("Shop"):
        with gr.Row():
            q = gr.Textbox(label="Search", placeholder="Search products, brands, categories", scale=3)
            cat = gr.Dropdown(["All"] + CATEGORIES, value="All", label="Category")
            sort = gr.Dropdown(["Relevance", "Price: low to high", "Price: high to low", "Top rated"], value="Relevance", label="Sort")
        with gr.Row():
            rating = gr.Slider(0, 5, value=0, step=0.5, label="Min rating")
            maxp = gr.Number(value=200000, label="Max price (Rs)")
        sug = gr.Markdown()
        count = gr.Markdown()
        grid = gr.Dataframe(value=table(PRODUCTS[:50]), interactive=False, wrap=True)
        gr.Markdown("**Click a row** (or type a product ID) to see details.")
        with gr.Row():
            pid = gr.Textbox(label="Product ID", scale=1)
            qty = gr.Number(value=1, minimum=1, precision=0, label="Quantity", scale=1)
            add_btn = gr.Button("Add to cart", variant="primary", scale=1)
            buy_btn = gr.Button("Buy now", scale=1)
        msg = gr.Markdown()
        with gr.Row():
            info = gr.Markdown(); gallery = gr.Gallery(label="Images", columns=3, height=300)
        filters = [q, cat, rating, maxp, sort]
        for c in filters: c.change(search, filters, [grid, count])
        q.change(lambda t: "Suggestions: " + " | ".join(suggest(t)) if t else "", q, sug)
        grid.select(pick, grid, pid)
        pid.change(detail, pid, [info, gallery])
        add_btn.click(add_to_cart, [pid, qty, cart], [cart, msg])
        buy_btn.click(add_to_cart, [pid, qty, cart], [cart, msg])

    with gr.Tab("Cart & Checkout") as cart_tab:
        cart_df = gr.Dataframe(interactive=False); total_md = gr.Markdown()
        with gr.Row():
            rm_id = gr.Textbox(label="Product ID to remove"); rm_btn = gr.Button("Remove")
        gr.Markdown("### Delivery address & payment")
        with gr.Row():
            name, line1 = gr.Textbox(label="Full name"), gr.Textbox(label="Address")
        with gr.Row():
            city, pin = gr.Textbox(label="City"), gr.Textbox(label="Pincode")
        pay = gr.Radio(["UPI", "Card", "Net Banking", "COD"], value="UPI", label="Payment method")
        place = gr.Button("Place order", variant="primary"); result = gr.Markdown()
        cart_tab.select(cart_view, cart, [cart_df, total_md])
        buy_btn.click(cart_view, cart, [cart_df, total_md])
        rm_btn.click(remove_from_cart, [rm_id, cart], cart).then(cart_view, cart, [cart_df, total_md])
        place.click(checkout, [cart, uid, name, line1, city, pin, pay], [cart, result]).then(cart_view, cart, [cart_df, total_md])

    with gr.Tab("Account & Orders"):
        with gr.Row():
            email = gr.Textbox(label="Email", value="user@shopease.com"); pw = gr.Textbox(label="Password", type="password", value="user1234")
        with gr.Row():
            login_btn, logout_btn = gr.Button("Log in", variant="primary"), gr.Button("Log out")
        status = gr.Markdown("Not logged in.")
        orders_btn = gr.Button("Show my orders"); orders_md = gr.Markdown()
        login_btn.click(login, [email, pw], [uid, status]); logout_btn.click(logout, None, [uid, status])
        orders_btn.click(orders_view, uid, orders_md)

    with gr.Tab("Support"):
        pol = gr.Dropdown(list(POLICIES), value="FAQ", label="Topic"); pol_md = gr.Markdown(policy_view("FAQ"))
        pol.change(policy_view, pol, pol_md)

    with gr.Tab("Chat"):
        gr.Markdown("Chatbot placeholder — wired to `POST /chat` when `CHAT_URL` is set in the previous cell.")
        gr.ChatInterface(chat_fn)

demo.launch(share=True, debug=True)